**we are gonna use pretrained model and finetune it for our dataset.
We are gonna use VGG16 pretrained on CIFAR 10**

**What is Transfer Learning?**

*It is a machine learning technique where a model is trained
 on one task is reused (partially or fully) for a different but related task.Instead
 of training a model from scratch, Which computationally expensive  and require large
 datasets, transfer learning leverages knowledge from pre-trained model to improve
italicized text learning efficiency and performance*

**Transfer Learning Works**
*   Pretraining on a Large Dataset

*   Fine Tuning for new task*



In [11]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

**Data Preparation**

In [12]:
# Cifar10 images are RGB images with 3x32x32 VGG expects 3x224x224
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [13]:
data_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
data_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

In [14]:
DataLoader_train = DataLoader(data_train, batch_size=32, shuffle=True)
DataLoader_test = DataLoader(data_test, batch_size=32, shuffle=False)

In [15]:
  # Import model
import torchvision.models as models
# Load pretrained model
model = models.vgg16(weights='IMAGENET1K_V1')

In [16]:
print(model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [17]:
#Freeze all layers
for param in model.features.parameters():
  param.requires_grad = False
# add your classifier
model.classifier[6] = nn.Linear(4096,10)



In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [23]:
epochs=10
lr  = 0.01
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr)


In [24]:
for epoch in range(epochs):
  for images,labels in DataLoader_train:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    output = model(images)
    loss_value = loss(output,labels)
    loss_value.backward()
    optimizer.step()
  print(f"Epoch [{epoch+1}/{epochs}] Loss: {loss_value.item():.4f}")

Epoch [1/10] Loss: 0.6688
Epoch [2/10] Loss: 0.2802
Epoch [3/10] Loss: 0.1455
Epoch [4/10] Loss: 0.2189
Epoch [5/10] Loss: 0.0706
Epoch [6/10] Loss: 0.2002
Epoch [7/10] Loss: 0.2331
Epoch [8/10] Loss: 0.0229
Epoch [9/10] Loss: 0.0377
Epoch [10/10] Loss: 0.0094


In [26]:
#Calculate accuracy
total = 0
correct = 0
with torch.no_grad():
  for images,labels in DataLoader_test:
    images = images.to(device)
    labels = labels.to(device)
    output = model(images)
    predicted = torch.argmax(output, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total
print(f'Accuracy: {accuracy:.2f}%')



Accuracy: 86.45%
